In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
real = '/content/drive/MyDrive/0000_Kaan_CDS'
link = '/content/drive/MyDrive/0000_Kaan_CDS'
if not os.path.exists(link):
    os.symlink(real, link)
print('Symlink:', os.path.exists(link))

In [ ]:
# CELL 1 — Setup & Imports
import importlib, sys, os
# --- Drive Mount ---
# from google.colab import drive
# drive.mount('/content/drive')
src_dir = '/content/drive/MyDrive/0000_Kaan_CDS/src'
os.chdir(src_dir)
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Core modules (M1-M4 required)
for mod_name in ['plot_style', 'cds_config', 'm1_calibration', 'm2_thresholds',
                  'm3_errors', 'm4_e2e']:
    spec = importlib.util.spec_from_file_location(mod_name, os.path.join(src_dir, f"{mod_name}.py"))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)
    print(f"  ✓ {mod_name}")

# Optional modules (M5, M6 — load if present, otherwise skip)
HAS_SHAP = False
HAS_CONFORMAL = False

for mod_name in ['m5_shap', 'm6_conformal']:
    fpath = os.path.join(src_dir, f"{mod_name}.py")
    if os.path.exists(fpath):
        try:
            spec = importlib.util.spec_from_file_location(mod_name, fpath)
            mod = importlib.util.module_from_spec(spec)
            sys.modules[mod_name] = mod
            spec.loader.exec_module(mod)
            if mod_name == 'm5_shap':    HAS_SHAP = True
            if mod_name == 'm6_conformal': HAS_CONFORMAL = True
            print(f"  ✓ {mod_name}")
        except Exception as e:
            print(f"  ⚠ {mod_name} import error: {e}")
    else:
        print(f"  ⚠ {mod_name}.py not found — optional, continuing")

from cds_config import *
setup_notebook("Chat 7 — M7 Cascade & Reflex")

from m1_calibration import load_calibrated_probs
from m2_thresholds import (load_threshold_config, apply_stage1_threshold,
                            assign_confidence_zone, get_operating_point)

if HAS_CONFORMAL:
    from m6_conformal import load_prediction_sets

import numpy as np
import pandas as pd
import json

print(f"\n{'='*50}")
print(f"Conformal: {'✓' if HAS_CONFORMAL else '✗'}")
print(f"SHAP:      {'✓' if HAS_SHAP else '✗'}")
print(f"{'='*50}")

## CELL 2 — Load all required data


In [ ]:
# CELL 2 — Load all required data
# 4 parquet: S1×{CBC_Only, CBC_BIO} + S2×{CBC_Only, CBC_BIO} — all test split

config = load_threshold_config(PATHS)

data = {}
for scenario in SCENARIOS:
    for stage in ['1', '2']:
        key = f"S{stage}_{scenario}"
        df = load_calibrated_probs(PATHS, stage, scenario, 'test')
        data[key] = df
        print(f"{key}: {len(df)} patients, columns: {df.shape[1]}")

# Conformal prediction sets (optional)
conformal = {}
if HAS_CONFORMAL:
    for scenario in SCENARIOS:
        for stage in ['1', '2']:
            key = f"S{stage}_{scenario}"
            try:
                alpha = 0.10
                cs = load_prediction_sets(PATHS, scenario, stage, alpha)
                conformal[key] = cs
                print(f"Conformal {key}: {len(cs)} patients")
            except Exception as e:
                print(f"Conformal {key}: ⚠ {e}")

# Sanity check: S1 and S2 patient counts
print(f"\n{'─'*50}")
print(f"S1 test: {len(data['S1_CBC_Only'])} patients (all)")
print(f"S2 test: {len(data['S2_CBC_Only'])} patients (AAC only)")
print(f"Difference (OAC patients): {len(data['S1_CBC_Only']) - len(data['S2_CBC_Only'])}")

## CELL 3 — Check index alignment across S1, S2, CBC_Only, CBC_BIO

In [ ]:
# CELL 3 — Check index alignment across S1, S2, CBC_Only, CBC_BIO
# We need to match patients for the cascade simulation

s1_cbc = data['S1_CBC_Only']
s2_cbc = data['S2_CBC_Only']
s1_bio = data['S1_CBC_BIO']
s2_bio = data['S2_CBC_BIO']

# Are the indices identical?
print("S1 CBC vs S1 BIO index match:", s1_cbc.index.equals(s1_bio.index))
print("S2 CBC vs S2 BIO index match:", s2_cbc.index.equals(s2_bio.index))
print("S2 index ⊂ S1 index:", s2_cbc.index.isin(s1_cbc.index).all())

# Target match check (same patient → same true label)
if s1_cbc.index.equals(s1_bio.index):
    print("S1 target match:", (s1_cbc['target'] == s1_bio['target']).all())
if s2_cbc.index.equals(s2_bio.index):
    print("S2 target match:", (s2_cbc['target'] == s2_bio['target']).all())

# Find true AAC patients in S1 — these should match the S2 patients
s1_ias_idx = s1_cbc.index[s1_cbc['target'] == 1]
print(f"\nTrue AAC in S1: {len(s1_ias_idx)}")
print(f"Patients in S2:  {len(s2_cbc)}")
print(f"Match: {s1_ias_idx.isin(s2_cbc.index).sum()}/{len(s1_ias_idx)}")

## CELL 4 — Find the S1↔S2 matching key

In [ ]:
# CELL 4 — Find the S1↔S2 matching key

s1_cbc = data['S1_CBC_Only']
s2_cbc = data['S2_CBC_Only']

# True AAC patients in S1
s1_ias = s1_cbc[s1_cbc['target'] == 1].copy()

print(f"S1 AAC: {len(s1_ias)}, S2: {len(s2_cbc)}")
print(f"Index match: {s1_ias.index.isin(s2_cbc.index).sum()}/165")

# Find common columns
common_cols = s1_ias.columns.intersection(s2_cbc.columns).tolist()
print(f"\nNumber of common columns: {len(common_cols)}")

# Is there a diagnosis column? Is it unique?
if 'diagnosis' in common_cols:
    print(f"\nS1 AAC diagnosis unique: {s1_ias['diagnosis'].nunique()}")
    print(f"S2 diagnosis unique: {s2_cbc['diagnosis'].nunique()}")
    print(f"S1 AAC diagnosis duplicated: {s1_ias['diagnosis'].duplicated().sum()}")
    print(f"S2 diagnosis duplicated: {s2_cbc['diagnosis'].duplicated().sum()}")

# Try feature-based matching: take a few common numeric columns
feature_cols = [c for c in common_cols
                if c not in ['target', 'diagnosis', 'pred', 'correct', 'confidence',
                             'calibration_method']
                and not c.startswith('prob_')]
print(f"\nCommon feature columns: {len(feature_cols)}")
print(f"First 5: {feature_cols[:5]}")

# Test matching with feature values
# Pick 3 features from S1 AAC, find rows with same values in S2
test_feats = feature_cols[:5]
s1_key = s1_ias[test_feats].round(8).astype(str).agg('|'.join, axis=1)
s2_key = s2_cbc[test_feats].round(8).astype(str).agg('|'.join, axis=1)

matched = s1_key.isin(s2_key.values)
print(f"\nMatch with 5-feature key: {matched.sum()}/{len(s1_ias)}")

# Try with all features
s1_key_full = s1_ias[feature_cols].round(8).astype(str).agg('|'.join, axis=1)
s2_key_full = s2_cbc[feature_cols].round(8).astype(str).agg('|'.join, axis=1)
matched_full = s1_key_full.isin(s2_key_full.values)
print(f"Match with full feature key: {matched_full.sum()}/{len(s1_ias)}")

## CELL 5 — Build the S1↔S2 index mapping table

In [ ]:
# CELL 5 — Build the S1↔S2 index mapping table

feature_cols = [c for c in data['S1_CBC_Only'].columns.intersection(data['S2_CBC_Only'].columns)
                if c not in ['target', 'diagnosis', 'pred', 'correct', 'confidence',
                             'calibration_method']
                and not c.startswith('prob_')]

s1_ias = data['S1_CBC_Only'][data['S1_CBC_Only']['target'] == 1].copy()

# S1 AAC → feature key
s1_key = s1_ias[feature_cols].round(8).astype(str).agg('|'.join, axis=1)
s2_key = data['S2_CBC_Only'][feature_cols].round(8).astype(str).agg('|'.join, axis=1)

# S2 key → S2 index mapping
s2_key_to_idx = dict(zip(s2_key.values, s2_key.index))

# S1 AAC index → S2 index
s1_to_s2 = {}
for s1_idx, key_val in s1_key.items():
    s2_idx = s2_key_to_idx.get(key_val)
    if s2_idx is not None:
        s1_to_s2[s1_idx] = s2_idx

print(f"S1→S2 mapping: {len(s1_to_s2)}/165")

# Validation: for already-matching indices, is the mapping consistent?
already_matched = sum(1 for k, v in s1_to_s2.items() if k == v)
print(f"Same index already: {already_matched}/165")
print(f"Different index: {len(s1_to_s2) - already_matched}/165")

# CELL 6 — Tier 1 → Tier 2 cascade simulation

In [ ]:
# CELL 6v2 — Corrected cascade (FP patients included)

s1_cbc = data['S1_CBC_Only']
s2_cbc = data['S2_CBC_Only']
s1_bio = data['S1_CBC_BIO']
s2_bio = data['S2_CBC_BIO']

cfg = load_threshold_config(PATHS)
ops_cbc = get_operating_point(cfg, 'CBC_Only')
ops_bio = get_operating_point(cfg, 'CBC_BIO')

S2_LABELS = {0: 'DEA', 1: 'HA', 2: 'HGB_HTZ', 3: 'Normal'}

rows = []
for idx in s1_cbc.index:
    row = {'patient_idx': idx}
    true_s1 = int(s1_cbc.loc[idx, 'target'])  # 0=DAS, 1=IAS

    # --- Ground truth ---
    if true_s1 == 0:
        row['true_label'] = 99
        row['true_label_name'] = 'DAS'
    else:
        s2_idx = s1_to_s2.get(idx)
        if s2_idx is not None and s2_idx in s2_cbc.index:
            row['true_label'] = int(s2_cbc.loc[s2_idx, 'target'])
            row['true_label_name'] = S2_LABELS[row['true_label']]
        else:
            row['true_label'] = np.nan
            row['true_label_name'] = 'UNKNOWN'

    # --- Tier 1: Stage 1 CBC ---
    prob_ias = s1_cbc.loc[idx, 'prob_IAS_cal']
    s1_pred = int(prob_ias >= ops_cbc['s1_threshold'])

    if s1_pred == 0:
        # DAS prediction → Excluded
        row['tier1_pred'] = 99
        row['tier1_pred_label'] = 'DAS'
        row['tier1_zone'] = 'Excluded'
        row['tier1_confidence'] = 1 - prob_ias
        row['escalated'] = True
        row['escalation_reason'] = 'excluded_DAS'
    else:
        # IAS prediction → enter S2 CBC
        s2_idx = s1_to_s2.get(idx)
        has_s2 = (s2_idx is not None and s2_idx in s2_cbc.index)

        if has_s2:
            # True IAS (or rare FP match) → S2 data exists
            prob_cols = ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']
            probs = s2_cbc.loc[s2_idx, prob_cols].values
            pred_class = int(np.argmax(probs))
            conf = float(np.max(probs))

            if conf >= ops_cbc['s2_zone_high']:
                zone = 'HIGH'
            elif conf < ops_cbc['s2_zone_low']:
                zone = 'LOW'
            else:
                zone = 'MEDIUM'

            row['tier1_pred'] = pred_class
            row['tier1_pred_label'] = S2_LABELS[pred_class]
            row['tier1_zone'] = zone
            row['tier1_confidence'] = conf
        else:
            # FP patient: S1 said IAS but truly DAS → no S2 data
            # In a real system S2 would return an IAS subtype (but wrong)
            # Since we have no S2 data, assign MEDIUM zone and escalate
            row['tier1_pred'] = -1  # unknown S2 result
            row['tier1_pred_label'] = 'FP_NO_S2'
            row['tier1_zone'] = 'MEDIUM'
            row['tier1_confidence'] = np.nan

        # Escalation decision
        if has_s2 and row['tier1_zone'] == 'HIGH':
            row['escalated'] = False
            row['escalation_reason'] = 'high_confidence'
        else:
            row['escalated'] = True
            reason = 'fp_no_s2_data' if not has_s2 else f"zone_{row['tier1_zone']}"
            row['escalation_reason'] = reason

    # --- Tier 2: CBC_BIO (escalated only) ---
    if row['escalated']:
        prob_ias_bio = s1_bio.loc[idx, 'prob_IAS_cal']
        s1_bio_pred = int(prob_ias_bio >= ops_bio['s1_threshold'])

        if s1_bio_pred == 0:
            # DAS with BIO
            row['tier2_pred'] = 99
            row['tier2_pred_label'] = 'DAS'
            row['tier2_zone'] = 'Excluded'
            row['tier2_confidence'] = 1 - prob_ias_bio
        else:
            # IAS → S2 BIO
            s2_idx = s1_to_s2.get(idx)
            has_s2_bio = (s2_idx is not None and s2_idx in s2_bio.index)

            if has_s2_bio:
                prob_cols_bio = ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']
                probs_bio = s2_bio.loc[s2_idx, prob_cols_bio].values
                pred_bio = int(np.argmax(probs_bio))
                conf_bio = float(np.max(probs_bio))

                if conf_bio >= ops_bio['s2_zone_high']:
                    zone_bio = 'HIGH'
                elif conf_bio < ops_bio['s2_zone_low']:
                    zone_bio = 'LOW'
                else:
                    zone_bio = 'MEDIUM'

                row['tier2_pred'] = pred_bio
                row['tier2_pred_label'] = S2_LABELS[pred_bio]
                row['tier2_zone'] = zone_bio
                row['tier2_confidence'] = conf_bio
            else:
                # FP patient: S1 BIO also said IAS but no S2 BIO data
                # This patient is truly DAS → whatever BIO S2 returns is wrong
                row['tier2_pred'] = -1
                row['tier2_pred_label'] = 'FP_NO_S2'
                row['tier2_zone'] = 'MEDIUM'
                row['tier2_confidence'] = np.nan
    else:
        row['tier2_pred'] = np.nan
        row['tier2_pred_label'] = np.nan
        row['tier2_zone'] = np.nan
        row['tier2_confidence'] = np.nan

    # --- Final decision ---
    if not row['escalated']:
        row['final_pred'] = row['tier1_pred']
        row['final_pred_label'] = row['tier1_pred_label']
        row['final_zone'] = row['tier1_zone']
        row['final_tier'] = 1
    else:
        row['final_pred'] = row.get('tier2_pred', np.nan)
        row['final_pred_label'] = row.get('tier2_pred_label', np.nan)
        row['final_zone'] = row.get('tier2_zone', np.nan)
        row['final_tier'] = 2

    # --- Correctness ---
    fp = row['final_pred']
    tl = row['true_label']
    if pd.isna(fp) or pd.isna(tl):
        row['correct'] = np.nan
    elif fp == -1:
        # FP patient: no S2 data, returned an IAS subtype but truly DAS → wrong
        row['correct'] = False
    else:
        row['correct'] = (fp == tl)

    rows.append(row)

cascade_df = pd.DataFrame(rows)

# ─── Summary ───
print(f"Total patients: {len(cascade_df)}")
print(f"\n── Tier 1 Zone Distribution ──")
print(cascade_df['tier1_zone'].value_counts())
print(f"\n── Escalation Reason ──")
print(cascade_df['escalation_reason'].value_counts())
print(f"\n── Escalation ──")
esc = cascade_df['escalated']
print(f"Escalated: {esc.sum()} ({esc.mean():.1%}), Tier 1 final: {(~esc).sum()} ({(~esc).mean():.1%})")
print(f"\n── Final Tier ──")
print(cascade_df['final_tier'].value_counts())
print(f"\n── Final Pred Distribution ──")
print(cascade_df['final_pred_label'].value_counts())

# ─── Accuracy ───
valid = cascade_df[cascade_df['correct'].notna()]
acc = valid['correct'].mean()
print(f"\n══ CASCADE E2E ACCURACY: {acc:.3f} ══")
print(f"Biochemistry requested: {esc.sum()}/{len(cascade_df)} ({esc.mean():.1%})")

# ─── M4 validation: CBC_Only E2E ───
# What would happen without escalation?
cbc_only_correct = 0
for _, r in cascade_df.iterrows():
    tl = r['true_label']
    if pd.isna(tl): continue
    # Use the Tier 1 result whatever it is
    if r['tier1_zone'] == 'Excluded':
        pred = 99  # DAS
    elif r['tier1_pred'] == -1:
        pred = -1  # FP → wrong
    else:
        pred = r['tier1_pred']
    if pred == tl:
        cbc_only_correct += 1

print(f"\nPure CBC_Only E2E (validation): {cbc_only_correct}/{len(valid)} = {cbc_only_correct/len(valid):.3f}  (expected ~0.555)")

## CELL 7 — Validation against M4 findings + different escalation strategies

###  CELL 7d — Count DAS false positives correctly

In [ ]:
# CELL 7d — Count DAS false positives correctly

s1_cbc = data['S1_CBC_Only']

# Take S1 pred from parquet (more reliable)
s1_preds = s1_cbc['pred'].values  # 0=DAS, 1=IAS

# E2E confusion analysis per patient
tp = fn = fp = tn = 0
s2_correct = 0
s2_total = 0

for i, idx in enumerate(s1_cbc.index):
    true_s1 = s1_cbc.loc[idx, 'target']  # 0=DAS, 1=IAS
    pred_s1 = s1_cbc.loc[idx, 'pred']

    if true_s1 == 0 and pred_s1 == 0: tn += 1
    elif true_s1 == 0 and pred_s1 == 1: fp += 1
    elif true_s1 == 1 and pred_s1 == 0: fn += 1
    else: tp += 1

print(f"═══ S1 CONFUSION ═══")
print(f"TP (IAS→IAS): {tp}")
print(f"TN (DAS→DAS): {tn}")
print(f"FP (DAS→IAS): {fp}  ← goes to S2, all will be wrong")
print(f"FN (IAS→DAS): {fn}  ← lost patient")
print(f"S1 accuracy: {(tp+tn)/(tp+tn+fp+fn):.3f}")

# True E2E: CBC_Only
# TN → correct DAS → correct
# FN → lost IAS (predicted DAS) → wrong
# FP → DAS but predicted IAS → goes to S2 → S2 returns IAS subtype → wrong (true=DAS)
# TP → IAS correct → goes to S2 → depends on S2 accuracy

# S2 result of TPs
tp_correct = 0
for idx in s1_cbc.index:
    if s1_cbc.loc[idx, 'target'] == 1 and s1_cbc.loc[idx, 'pred'] == 1:
        # True IAS, predicted IAS → enters S2
        s2_idx = s1_to_s2.get(idx)
        if s2_idx is not None and s2_idx in s2_cbc.index:
            s2_pred = int(np.argmax(s2_cbc.loc[s2_idx,
                ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']].values))
            s2_true = int(s2_cbc.loc[s2_idx, 'target'])
            if s2_pred == s2_true:
                tp_correct += 1
            s2_total += 1

print(f"\n═══ S2 (TP patients only) ═══")
print(f"S2 correct: {tp_correct}/{s2_total} ({tp_correct/s2_total:.3f})")

# True E2E accuracy
e2e_correct = tn + tp_correct  # TN correct + S2 correct
e2e_total = len(s1_cbc)
print(f"\n═══ TRUE CBC_Only E2E ═══")
print(f"Correct: {e2e_correct}/{e2e_total}")
print(f"  TN (correct DAS):    {tn}")
print(f"  TP → S2 correct:     {tp_correct}")
print(f"  FP (DAS→S2→wrong):   {fp}")
print(f"  FN (lost IAS):       {fn}")
print(f"E2E accuracy: {e2e_correct/e2e_total:.3f}  (M4 expected: 0.559)")

## CELL 8 — Different escalation strategies + FP analysis

In [ ]:
# CELL 8v2 — Compute CBC_Only and BIO predictions separately per patient

s1_cbc = data['S1_CBC_Only']
s2_cbc = data['S2_CBC_Only']
s1_bio = data['S1_CBC_BIO']
s2_bio = data['S2_CBC_BIO']

S2_LABELS = {0: 'DEA', 1: 'HA', 2: 'HGB_HTZ', 3: 'Normal'}

def compute_e2e_pred(idx, s1_df, s2_df, ops, s1_to_s2):
    """Compute the E2E prediction for a single patient"""
    prob_ias = s1_df.loc[idx, 'prob_IAS_cal']
    s1_pred = int(prob_ias >= ops['s1_threshold'])

    if s1_pred == 0:
        return 99, 'DAS', 1 - prob_ias  # DAS

    s2_idx = s1_to_s2.get(idx)
    if s2_idx is not None and s2_idx in s2_df.index:
        prob_cols = ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']
        probs = s2_df.loc[s2_idx, prob_cols].values
        pred = int(np.argmax(probs))
        conf = float(np.max(probs))
        return pred, S2_LABELS[pred], conf
    else:
        # FP: S1 said IAS but no S2 data → unknown subtype
        # In a real system it would predict a subtype but we don't know it
        return -1, 'FP_NO_S2', prob_ias

# Two-scenario prediction per patient
rows = []
for idx in s1_cbc.index:
    row = {'patient_idx': idx}

    # Ground truth
    true_s1 = int(s1_cbc.loc[idx, 'target'])
    if true_s1 == 0:
        row['true_label'] = 99
        row['true_label_name'] = 'DAS'
    else:
        s2_idx = s1_to_s2.get(idx)
        if s2_idx is not None and s2_idx in s2_cbc.index:
            row['true_label'] = int(s2_cbc.loc[s2_idx, 'target'])
            row['true_label_name'] = S2_LABELS[row['true_label']]
        else:
            row['true_label'] = np.nan
            row['true_label_name'] = 'UNKNOWN'

    # CBC_Only E2E
    cbc_pred, cbc_label, cbc_conf = compute_e2e_pred(idx, s1_cbc, s2_cbc, ops_cbc, s1_to_s2)
    row['cbc_pred'] = cbc_pred
    row['cbc_pred_label'] = cbc_label
    row['cbc_conf'] = cbc_conf

    # CBC_BIO E2E
    bio_pred, bio_label, bio_conf = compute_e2e_pred(idx, s1_bio, s2_bio, ops_bio, s1_to_s2)
    row['bio_pred'] = bio_pred
    row['bio_pred_label'] = bio_label
    row['bio_conf'] = bio_conf

    # Tier 1 zone (CBC_Only S2 result)
    prob_ias_cbc = s1_cbc.loc[idx, 'prob_IAS_cal']
    s1_pred_cbc = int(prob_ias_cbc >= ops_cbc['s1_threshold'])

    if s1_pred_cbc == 0:
        row['tier1_zone'] = 'Excluded'
    elif cbc_pred == -1:
        row['tier1_zone'] = 'MEDIUM'  # FP, no S2 data
    else:
        if cbc_conf >= ops_cbc['s2_zone_high']:
            row['tier1_zone'] = 'HIGH'
        elif cbc_conf < ops_cbc['s2_zone_low']:
            row['tier1_zone'] = 'LOW'
        else:
            row['tier1_zone'] = 'MEDIUM'

    rows.append(row)

strat_df = pd.DataFrame(rows)

# ─── Compute correctness ───
def calc_correct(pred, true_label):
    if pd.isna(pred) or pd.isna(true_label):
        return np.nan
    if pred == -1:
        return False  # FP, wrong
    return pred == true_label

strat_df['cbc_correct'] = strat_df.apply(lambda r: calc_correct(r['cbc_pred'], r['true_label']), axis=1)
strat_df['bio_correct'] = strat_df.apply(lambda r: calc_correct(r['bio_pred'], r['true_label']), axis=1)

# ─── Validation ───
print(f"═══ BASELINE VALIDATION ═══")
cbc_acc = strat_df['cbc_correct'].mean()
bio_acc = strat_df['bio_correct'].mean()
print(f"CBC_Only E2E: {cbc_acc:.3f}  (M4 expected: 0.559)")
print(f"CBC_BIO E2E:  {bio_acc:.3f}  (M4 expected: 0.699)")

print(f"\n═══ ZONE DISTRIBUTION ═══")
print(strat_df['tier1_zone'].value_counts())

# ─── Strategy simulations ───
def run_strategy(df, escalate_mask, name):
    """escalate_mask: patients flagged True use the BIO result"""
    preds = np.where(escalate_mask, df['bio_pred'], df['cbc_pred'])
    true_labels = df['true_label'].values

    valid = ~(pd.isna(true_labels))
    correct = 0
    for i in range(len(df)):
        if not valid[i]: continue
        p = preds[i]
        t = true_labels[i]
        if p == -1:
            pass  # FP → wrong
        elif p == t:
            correct += 1

    total = valid.sum()
    acc = correct / total
    bio_count = escalate_mask.sum()
    bio_pct = bio_count / total * 100
    return {'strategy': name, 'accuracy': acc, 'bio_pct': bio_pct, 'bio_count': int(bio_count)}

strategies = [
    ('Pure CBC_Only (0% bio)',
     pd.Series(False, index=strat_df.index)),

    ('LOW only → BIO',
     strat_df['tier1_zone'] == 'LOW'),

    ('Excluded only → BIO',
     strat_df['tier1_zone'] == 'Excluded'),

    ('MEDIUM + LOW → BIO',
     strat_df['tier1_zone'].isin(['MEDIUM', 'LOW'])),

    ('Excluded + MEDIUM + LOW → BIO',
     strat_df['tier1_zone'].isin(['Excluded', 'MEDIUM', 'LOW'])),

    ('Full BIO (100%)',
     pd.Series(True, index=strat_df.index)),
]

print(f"\n═══ STRATEGY COMPARISON ═══")
print(f"{'Strategy':<35} {'Acc':>6} {'Bio%':>6} {'Bio#':>5}")
print("─" * 55)

results = []
for name, mask in strategies:
    res = run_strategy(strat_df, mask, name)
    results.append(res)
    print(f"{res['strategy']:<35} {res['accuracy']:>6.3f} {res['bio_pct']:>5.1f}% {res['bio_count']:>5}")

results_df = pd.DataFrame(results)
baseline = results_df.iloc[0]['accuracy']
full_bio = results_df.iloc[-1]['accuracy']
gain_range = full_bio - baseline

results_df['gain_pct'] = ((results_df['accuracy'] - baseline) / gain_range * 100).round(1)

print(f"\n═══ GAIN ═══")
print(f"{'Strategy':<35} {'Acc':>6} {'Bio%':>6} {'Gain':>8}")
print("─" * 58)
for _, r in results_df.iterrows():
    print(f"{r['strategy']:<35} {r['accuracy']:>6.3f} {r['bio_pct']:>5.1f}% {r['gain_pct']:>7.1f}%")

## CELL 9 — Continuous efficiency curve (HIGH threshold sweep)

In [ ]:
# CELL 9 — Continuous efficiency curve (HIGH threshold sweep)
# Sweeping the HIGH threshold from high to low:
# - High threshold → few patients HIGH → more escalation → more bio
# - Low threshold → many patients HIGH → less escalation → less bio

thresholds = np.arange(0.25, 1.01, 0.01)
curve_data = []

for thr in thresholds:
    preds = []
    bio_count = 0

    for _, r in strat_df.iterrows():
        # Recompute the zone with this threshold
        if r['tier1_zone'] == 'Excluded':
            # Excluded always escalates
            zone = 'Excluded'
        elif pd.isna(r['cbc_conf']):
            # FP patient
            zone = 'MEDIUM'
        else:
            zone = 'HIGH' if r['cbc_conf'] >= thr else 'MEDIUM'

        # Escalation decision
        if zone == 'HIGH':
            # Tier 1 final → CBC_Only result
            preds.append(r['cbc_pred'])
        else:
            # Escalation → BIO result
            bio_count += 1
            preds.append(r['bio_pred'])

    # Compute accuracy
    preds = np.array(preds, dtype=float)
    true_labels = strat_df['true_label'].values
    valid = ~np.isnan(true_labels)

    correct = 0
    for i in range(len(strat_df)):
        if not valid[i]: continue
        p = preds[i]
        t = true_labels[i]
        if p == -1:
            pass  # FP → wrong
        elif p == t:
            correct += 1

    total = valid.sum()
    acc = correct / total
    bio_pct = bio_count / total * 100

    curve_data.append({
        'threshold': thr,
        'accuracy': acc,
        'bio_pct': bio_pct,
        'bio_count': bio_count
    })

curve_df = pd.DataFrame(curve_data)

# Also without including Excluded (only IAS MEDIUM→BIO)
curve_no_excl = []
for thr in thresholds:
    preds = []
    bio_count = 0

    for _, r in strat_df.iterrows():
        if r['tier1_zone'] == 'Excluded':
            # Excluded → CBC_Only result (no escalation)
            preds.append(r['cbc_pred'])
        elif pd.isna(r['cbc_conf']):
            # FP → CBC_Only result
            preds.append(r['cbc_pred'])
        else:
            zone = 'HIGH' if r['cbc_conf'] >= thr else 'MEDIUM'
            if zone == 'HIGH':
                preds.append(r['cbc_pred'])
            else:
                bio_count += 1
                preds.append(r['bio_pred'])

    preds = np.array(preds, dtype=float)
    true_labels = strat_df['true_label'].values
    valid = ~np.isnan(true_labels)
    correct = sum(1 for i in range(len(strat_df))
                  if valid[i] and preds[i] != -1 and preds[i] == true_labels[i])
    total = valid.sum()

    curve_no_excl.append({
        'threshold': thr,
        'accuracy': correct / total,
        'bio_pct': bio_count / total * 100,
        'bio_count': bio_count
    })

curve_no_excl_df = pd.DataFrame(curve_no_excl)

# ─── Result ───
print("═══ EFFICIENCY CURVE SUMMARY ═══")
print(f"\nExcluded INCLUDED (Excl→BIO):")
print(f"  Min bio: {curve_df['bio_pct'].min():.1f}% → acc={curve_df.loc[curve_df['bio_pct'].idxmin(), 'accuracy']:.3f}")
print(f"  Max bio: {curve_df['bio_pct'].max():.1f}% → acc={curve_df.loc[curve_df['bio_pct'].idxmax(), 'accuracy']:.3f}")

print(f"\nExcluded EXCLUDED (Excl→stays CBC):")
print(f"  Min bio: {curve_no_excl_df['bio_pct'].min():.1f}% → acc={curve_no_excl_df.loc[curve_no_excl_df['bio_pct'].idxmin(), 'accuracy']:.3f}")
print(f"  Max bio: {curve_no_excl_df['bio_pct'].max():.1f}% → acc={curve_no_excl_df.loc[curve_no_excl_df['bio_pct'].idxmax(), 'accuracy']:.3f}")

# Sweet spot search: highest accuracy around 30% bio
sweet_zone = curve_no_excl_df[(curve_no_excl_df['bio_pct'] >= 25) & (curve_no_excl_df['bio_pct'] <= 35)]
if len(sweet_zone) > 0:
    best = sweet_zone.loc[sweet_zone['accuracy'].idxmax()]
    print(f"\n═══ SWEET SPOT (25-35% bio range) ═══")
    print(f"  Threshold: {best['threshold']:.2f}")
    print(f"  Bio: {best['bio_pct']:.1f}%")
    print(f"  Accuracy: {best['accuracy']:.3f}")
    gain = (best['accuracy'] - 0.559) / (0.699 - 0.559) * 100
    print(f"  Gain: {gain:.1f}%")
else:
    print(f"\n⚠ No strategy found in the 25-35% range")
    # Find the closest to 30%
    closest = curve_no_excl_df.iloc[(curve_no_excl_df['bio_pct'] - 30).abs().argsort()[:3]]
    print(f"Closest to 30%:")
    print(closest[['threshold', 'accuracy', 'bio_pct']].to_string())

## CELL 10 — Examine the 25-35% range on both curves in detail

In [ ]:
# CELL 10 — Examine the 25-35% range on both curves + reconstruct the M4 strategy

# 25-35% range on the Excluded-INCLUDED curve
sweet_incl = curve_df[(curve_df['bio_pct'] >= 25) & (curve_df['bio_pct'] <= 35)]
if len(sweet_incl) > 0:
    best_incl = sweet_incl.loc[sweet_incl['accuracy'].idxmax()]
    gain_incl = (best_incl['accuracy'] - 0.559) / (0.699 - 0.559) * 100
    print(f"═══ Excluded INCLUDED — 25-35% range ═══")
    print(f"  Threshold: {best_incl['threshold']:.2f}")
    print(f"  Bio: {best_incl['bio_pct']:.1f}%  Acc: {best_incl['accuracy']:.3f}  Gain: {gain_incl:.1f}%")
else:
    print("No 25-35% range on the Excluded-INCLUDED curve (starts at min 14.4%)")
    # Show the closest points
    print("\nExcluded-INCLUDED curve — all points (first 20):")
    print(curve_df[['threshold', 'accuracy', 'bio_pct']].head(20).to_string())

# ─── Reconstruct the M4 strategy ───
# M4 had "Strategy 4: Selective biochemistry"
# Likely: do CBC_Only E2E → add BIO for the uncertain ones
# i.e. HIGH → CBC result, rest → BIO result
# But how does 29% come out?

# Possibility: M4 computed 29% over IAS (DAS patients are already correct)
# IAS = 165 patients, 29% = ~48 patients
ias_patients = strat_df[strat_df['true_label'] != 99]
print(f"\n═══ M4 RECONSTRUCTION ═══")
print(f"Number of AAC patients: {len(ias_patients)}")
print(f"29% of AAC: {len(ias_patients) * 0.29:.0f} patients")

# Zone distribution within AAC patients
ias_zones = strat_df[strat_df['true_label'] != 99]['tier1_zone'].value_counts()
print(f"\nAAC patient zone distribution:")
print(ias_zones)
for zone, cnt in ias_zones.items():
    print(f"  {zone}: {cnt} ({cnt/len(ias_patients):.1%})")

# Hypothesis: M4 computed only over IAS patients entering S2
# Those predicted IAS at S1 = those entering S2
ias_in_s2 = strat_df[(strat_df['tier1_zone'] != 'Excluded') & (strat_df['true_label'] != 99)]
print(f"\nAAC & entering S2 (excluding Excluded): {len(ias_in_s2)}")
print(f"  HIGH: {(ias_in_s2['tier1_zone'] == 'HIGH').sum()}")
print(f"  MEDIUM: {(ias_in_s2['tier1_zone'] == 'MEDIUM').sum()}")

# HIGH IAS patients: no biochemistry needed
# MEDIUM IAS patients: biochemistry needed
high_ias = ias_in_s2[ias_in_s2['tier1_zone'] == 'HIGH']
med_ias = ias_in_s2[ias_in_s2['tier1_zone'] == 'MEDIUM']
print(f"\n  HIGH AAC acc (CBC_Only): {high_ias['cbc_correct'].mean():.3f}")
print(f"  MEDIUM AAC: biochemistry needed → {len(med_ias)} patients")
print(f"  MEDIUM AAC / all patients: {len(med_ias)/len(strat_df):.1%}")
print(f"  MEDIUM AAC / AAC patients: {len(med_ias)/len(ias_patients):.1%}")

# ─── Fully replicate the M4 calculation ───
# Hypothesis: M4 "patient ratio" = biochemistry requested / TOTAL (229)
# Strategy: HIGH IAS → CBC result, rest IAS → BIO
# DAS patients: predicted DAS at S1 → already correct DAS
#               predicted IAS at S1 → FP, wrong

# Send only S2 MEDIUM IAS to BIO (Excluded and FP stay in CBC)
mask_m4 = (strat_df['tier1_zone'] == 'MEDIUM') & (strat_df['cbc_pred'] != -1)
res_m4 = run_strategy(strat_df, mask_m4, 'M4-style: only S2 MEDIUM AAC→BIO')
gain_m4 = (res_m4['accuracy'] - 0.559) / (0.699 - 0.559) * 100
print(f"\n═══ M4-STYLE STRATEGY ═══")
print(f"  Acc: {res_m4['accuracy']:.3f}")
print(f"  Bio: {res_m4['bio_pct']:.1f}%")
print(f"  Bio#: {res_m4['bio_count']}")
print(f"  Gain: {gain_m4:.1f}%")

## CELL 10b — A different approach to understand the M4 calculation

In [ ]:
# CELL 10b — A different approach to understand the M4 calculation
# M4 likely sorted by confidence and added incrementally

# Possibility: M4 sorted all patients by CBC_Only confidence
# Lowest confidence → send to BIO, move upward
# 29% patients → BIO gave 94% gain

# Approach: send each patient in confidence order
patient_list = strat_df.copy()

# For each patient: correct if sent to BIO vs correct if kept in CBC?
patient_list['cbc_correct_bool'] = patient_list['cbc_correct'].fillna(False).astype(bool)
patient_list['bio_correct_bool'] = patient_list['bio_correct'].fillna(False).astype(bool)

# CBC confidence: S1 confidence for Excluded, S2 confidence for the rest
# Lower confidence = more need for BIO
patient_list['sort_conf'] = patient_list['cbc_conf']
# Excluded and FP: cbc_conf already exists
# Put NaN ones at the lowest
patient_list['sort_conf'] = patient_list['sort_conf'].fillna(0)

# Confidence ASCENDING (most uncertain to BIO first)
patient_list = patient_list.sort_values('sort_conf', ascending=True).reset_index(drop=True)

# Incremental sweep: send patients 0..n to BIO
n = len(patient_list)
sweep_results = []

for k in range(n + 1):
    bio_mask = np.zeros(n, dtype=bool)
    bio_mask[:k] = True  # first k patients to BIO

    correct = 0
    for i in range(n):
        if bio_mask[i]:
            if patient_list.iloc[i]['bio_correct_bool']:
                correct += 1
        else:
            if patient_list.iloc[i]['cbc_correct_bool']:
                correct += 1

    acc = correct / n
    bio_pct = k / n * 100
    sweep_results.append({
        'k': k, 'accuracy': acc, 'bio_pct': bio_pct,
        'conf_threshold': patient_list.iloc[k-1]['sort_conf'] if k > 0 else 0
    })

sweep_df = pd.DataFrame(sweep_results)

# Find M4 reference points
print("═══ INCREMENTAL SWEEP RESULTS ═══")
print(f"{'k':>4} {'Bio%':>6} {'Acc':>6} {'Gain%':>9}")
print("─" * 30)

for target_pct in [0, 10, 20, 29, 30, 40, 50, 60, 70, 80, 90, 100]:
    closest = sweep_df.iloc[(sweep_df['bio_pct'] - target_pct).abs().argsort()[:1]]
    r = closest.iloc[0]
    gain = (r['accuracy'] - 0.559) / (0.699 - 0.559) * 100 if 0.699 > 0.559 else 0
    print(f"{int(r['k']):>4} {r['bio_pct']:>5.1f}% {r['accuracy']:>6.3f} {gain:>8.1f}%")

# Sweet spot: closest to 94% gain
target_gain = 0.559 + 0.94 * (0.699 - 0.559)  # = 0.6906
closest_sweet = sweep_df.iloc[(sweep_df['accuracy'] - target_gain).abs().argsort()[:1]]
r = closest_sweet.iloc[0]
print(f"\n═══ 94% GAIN POINT ═══")
print(f"  k={int(r['k'])}, Bio={r['bio_pct']:.1f}%, Acc={r['accuracy']:.3f}")
print(f"  Conf threshold: {r['conf_threshold']:.3f}")
gain = (r['accuracy'] - 0.559) / (0.699 - 0.559) * 100
print(f"  Actual gain: {gain:.1f}%")

## CELL 11 — Efficiency curve figure (publication quality)

In [ ]:
# CELL 11 — Efficiency curve figure (publication quality)

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

fig, ax = plt.subplots(figsize=(8, 5))

# ─── Main curve (incremental sweep) ───
ax.plot(sweep_df['bio_pct'], sweep_df['accuracy'],
        color=PALETTE['base1'], linewidth=2, zorder=3)

# ─── Reference points ───
ref_points = {
    'Pure CBC\n(0%)': (0, 0.559),
    'Sweet spot\n(~70%)': (69.9, 0.690),
    'Full BIO\n(100%)': (100, 0.699),
}

markers = {
    'Pure CBC\n(0%)': {'color': PALETTE['highlight'], 'marker': 's'},
    'Sweet spot\n(~70%)': {'color': PALETTE['accent1'], 'marker': 'D'},
    'Full BIO\n(100%)': {'color': PALETTE['accent2'], 'marker': 'o'},
}

for label, (x, y) in ref_points.items():
    m = markers[label]
    ax.scatter(x, y, color=m['color'], s=100, marker=m['marker'],
              zorder=5, edgecolors='white', linewidth=1.5)

# ─── Annotation: sweet spot ───
ax.annotate(
    'Biochemistry for 70% of patients\n→ 94% of the gain',
    xy=(69.9, 0.690), xytext=(40, 0.72),
    fontsize=9, fontweight='bold', color=PALETTE['accent1'],
    arrowprops=dict(arrowstyle='->', color=PALETTE['accent1'], lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=PALETTE['accent1'], alpha=0.9)
)

# ─── Reference lines ───
ax.axhline(y=0.559, color=PALETTE['highlight'], linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(y=0.699, color=PALETTE['accent2'], linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(y=0.690, color=PALETTE['accent1'], linestyle=':', alpha=0.4, linewidth=1)

# ─── Gain band ───
ax.fill_between([0, 100], 0.559, 0.699, color=PALETTE['base2'], alpha=0.1)

# ─── Cascade point (full cascade) ───
ax.scatter(83.4, 0.703, color=PALETTE['accent3'], s=80, marker='^',
          zorder=5, edgecolors='white', linewidth=1.5)
ax.annotate('Full Cascade\n(0.703)', xy=(83.4, 0.703), xytext=(88, 0.715),
           fontsize=8, color=PALETTE['accent3'],
           arrowprops=dict(arrowstyle='->', color=PALETTE['accent3'], lw=1))

# ─── Axis settings ───
ax.set_xlabel('Biochemistry Utilization (%)', fontsize=11)
ax.set_ylabel('End-to-End Accuracy', fontsize=11)
ax.set_xlim(-2, 105)
ax.set_ylim(0.54, 0.73)
ax.xaxis.set_major_formatter(PercentFormatter(100, decimals=0))

despine_all(ax)
fig.tight_layout()

# Save
out_dir = PATHS.module_dir('m7_cascade', 'plots')
save_fig(fig, out_dir, 'efficiency_curve')
plt.show()

print(f"\n✓ Figure saved: {out_dir}")

## CELL 12 — Lost patient analysis

In [ ]:
# CELL 12 — Lost patient analysis

# ─── 1. True AAC patients missed at Tier 1 ───
ias_patients = strat_df[strat_df['true_label'] != 99].copy()
ias_excluded = ias_patients[ias_patients['tier1_zone'] == 'Excluded']

print(f"═══ LOST PATIENT ANALYSIS ═══")
print(f"True AAC: {len(ias_patients)}")
print(f"Predicted OAC at S1 CBC (lost): {len(ias_excluded)} ({len(ias_excluded)/len(ias_patients):.1%})")
print(f"\nTrue classes of lost patients:")
print(ias_excluded['true_label_name'].value_counts())

# ─── 2. Tier 2 rescue analysis ───
# Take escalated AAC patients from cascade_df
ias_cascade = cascade_df[cascade_df['true_label'] != 99].copy()
ias_escalated = ias_cascade[ias_cascade['escalated'] == True]

# Correctly predicted at Tier 2 = rescued
ias_escalated_correct = ias_escalated[ias_escalated['correct'] == True]
ias_escalated_wrong = ias_escalated[ias_escalated['correct'] == False]

print(f"\n═══ TIER 2 RESCUE ═══")
print(f"Escalated AAC: {len(ias_escalated)}")
print(f"  Correct at Tier 2 (rescued): {len(ias_escalated_correct)} ({len(ias_escalated_correct)/len(ias_escalated):.1%})")
print(f"  Wrong at Tier 2:             {len(ias_escalated_wrong)} ({len(ias_escalated_wrong)/len(ias_escalated):.1%})")

# ─── 3. Tier 2 fate of Excluded AAC ───
excl_ias = cascade_df[(cascade_df['true_label'] != 99) & (cascade_df['escalation_reason'] == 'excluded_DAS')]
print(f"\n═══ EXCLUDED AAC → TIER 2 FATE ═══")
print(f"AAC missed at S1 CBC: {len(excl_ias)}")
if len(excl_ias) > 0:
    excl_rescued = excl_ias[excl_ias['correct'] == True]
    print(f"  Rescued with Tier 2 BIO: {len(excl_rescued)}")
    print(f"  Still lost:              {len(excl_ias) - len(excl_rescued)}")
    print(f"\n  Tier 2 prediction distribution:")
    print(f"  {excl_ias['tier2_pred_label'].value_counts().to_string()}")

# ─── 4. Per-class loss rate ───
DISP = {'DEA': 'IDA', 'HA': 'HA', 'HGB_HTZ': 'HGB HTZ', 'Normal': 'Normal'}
print(f"\n═══ PER-CLASS LOSS ANALYSIS ═══")
print(f"{'Class':<10} {'Total':>7} {'T1 HIGH':>8} {'Escal.':>7} {'T2 Correct':>11} {'Final Correct':>14} {'Lost':>6}")
print("─" * 70)

for cls_id, cls_name in S2_LABELS.items():
    cls_mask = ias_cascade['true_label'] == cls_id
    cls_df = ias_cascade[cls_mask]
    n_total = len(cls_df)
    if n_total == 0: continue

    n_high = ((cls_df['escalated'] == False)).sum()
    n_escalated = (cls_df['escalated'] == True).sum()
    n_t2_correct = ((cls_df['escalated'] == True) & (cls_df['correct'] == True)).sum()
    n_final_correct = (cls_df['correct'] == True).sum()
    n_lost = n_total - n_final_correct

    print(f"{DISP[cls_name]:<10} {n_total:>7} {n_high:>8} {n_escalated:>7} {n_t2_correct:>11} {n_final_correct:>14} {n_lost:>6} ({n_lost/n_total:.0%})")

# ─── 5. Conformal set vs loss risk correlation ───
if HAS_CONFORMAL and 'S2_CBC_Only' in conformal:
    print(f"\n═══ CONFORMAL SET × LOSS RISK ═══")
    conf_s2 = conformal['S2_CBC_Only']
    print(f"Conformal S2 CBC_Only columns: {list(conf_s2.columns)[:10]}")
    # Match and analyze
    # (conformal index should match s2_cbc)
else:
    print(f"\n⚠ Conformal data not available or could not be matched — skipping")

## CELL 13a — Check conformal columns

In [ ]:
# CELL 13a — Check conformal columns
if HAS_CONFORMAL:
    cs = conformal.get('S2_CBC_Only')
    if cs is not None:
        print("Conformal S2 CBC_Only ALL columns:")
        print(cs.columns.tolist())
        # Any column with set_size or starting with in_?
        set_cols = [c for c in cs.columns if 'set' in c.lower() or c.startswith('in_')]
        print(f"\nSet-related columns: {set_cols}")

## CELL 13b — Refleks Test Kural Motoru

In [ ]:
# CELL 13b — Refleks Test Kural Motoru
# (Reflex Test Recommendation Engine)
# NOTE: Internal prediction keys ("DEA","DAS",...) are kept unchanged so they match
# the cascade_df labels (S2_LABELS). Only the user-facing 'test'/'rationale' strings
# are in English. DISPLAY_LABEL maps internal codes to publication labels for figures.

# ─── Display label mapping (internal code -> publication label) ───
DISPLAY_LABEL = {
    "DEA": "IDA",        # iron deficiency anemia
    "HA": "HA",          # hemolytic anemia
    "HGB_HTZ": "HGB HTZ",
    "Normal": "Normal",
    "DAS": "OAC",        # other anemia causes
    "IAS": "AAC",        # anemia-associated causes
}

# ─── Rule definitions ───
reflex_rules = {
    "version": "M7_v2",
    "description": "Cascade-based reflex test recommendations",
    "rules": [
        # Tier 1 HIGH zone — confirmation tests
        {"prediction": "DEA", "zone": "HIGH", "tier": 1,
         "test": "Ferritin + serum iron", "urgency": "routine",
         "rationale": "IDA confirmation: iron deficiency parameters"},
        {"prediction": "HA", "zone": "HIGH", "tier": 1,
         "test": "LD + indirect bilirubin + haptoglobin + peripheral smear", "urgency": "urgent",
         "rationale": "Hemolysis confirmation: acute hemolysis risk"},
        {"prediction": "HGB_HTZ", "zone": "HIGH", "tier": 1,
         "test": "Hemoglobin electrophoresis / HPLC", "urgency": "routine",
         "rationale": "Hemoglobinopathy confirmation"},
        {"prediction": "Normal", "zone": "HIGH", "tier": 1,
         "test": "No additional testing required", "urgency": "none",
         "rationale": "High-confidence normal — follow-up sufficient"},
        # Tier 1 MEDIUM/LOW — escalation
        {"prediction": "*", "zone": "MEDIUM", "tier": 1,
         "test": "Biochemistry panel (Ferritin, Iron, LD, UIBC)", "urgency": "routine",
         "rationale": "Moderate confidence — escalate to Tier 2 with biochemistry"},
        {"prediction": "*", "zone": "LOW", "tier": 1,
         "test": "Biochemistry panel + specialist consultation", "urgency": "urgent",
         "rationale": "Low confidence — urgent biochemistry and expert review"},
        # Tier 1 Excluded — escalation
        {"prediction": "DAS", "zone": "Excluded", "tier": 1,
         "test": "Biochemistry panel (Ferritin, Iron, LD, UIBC)", "urgency": "routine",
         "rationale": "S1 OAC prediction — verify with biochemistry"},
        # Tier 2 results
        {"prediction": "DEA", "zone": "HIGH", "tier": 2,
         "test": "Evaluate for oral iron therapy; consider 3-month follow-up CBC", "urgency": "routine",
         "rationale": "Confirmed IDA — recommend clinical evaluation for iron therapy"},
        {"prediction": "HA", "zone": "HIGH", "tier": 2,
         "test": "Hemolysis panel (LD, indirect bilirubin, haptoglobin) + hematology consultation", "urgency": "urgent",
         "rationale": "Confirmed hemolytic anemia — investigate etiology"},
        {"prediction": "HGB_HTZ", "zone": "HIGH", "tier": 2,
         "test": "Hemoglobin electrophoresis / HPLC + genetic counseling", "urgency": "routine",
         "rationale": "Confirmed hemoglobinopathy carrier — family screening"},
        {"prediction": "Normal", "zone": "HIGH", "tier": 2,
         "test": "No additional testing required", "urgency": "none",
         "rationale": "Confirmed normal with full biochemistry panel"},
        {"prediction": "DAS", "zone": "Excluded", "tier": 2,
         "test": "Investigate other anemia etiologies", "urgency": "routine",
         "rationale": "OAC after biochemistry — consider chronic disease, B12/folate, renal causes"},
        {"prediction": "*", "zone": "MEDIUM", "tier": 2,
         "test": "Specialist review + extended workup", "urgency": "routine",
         "rationale": "Still uncertain after CBC+biochemistry — human verification needed"},
        {"prediction": "*", "zone": "LOW", "tier": 2,
         "test": "Specialist consultation + extended workup", "urgency": "urgent",
         "rationale": "Low confidence after biochemistry — specialist review required"},
    ]
}

# ─── Rule matching function ───
def get_reflex_recommendation(pred_label, zone, tier, rules):
    """Find the most appropriate reflex test rule for a patient."""
    for rule in rules['rules']:
        if rule['tier'] != tier:
            continue
        zone_match = (rule['zone'] == zone)
        pred_match = (rule['prediction'] == pred_label or rule['prediction'] == '*')
        if zone_match and pred_match:
            return rule
    return None

# ─── Her hastaya refleks test ata (Assign reflex test to each patient) ───
reflex_rows = []
for _, r in cascade_df.iterrows():
    rec = {}
    rec['patient_idx'] = r['patient_idx']
    rec['final_tier'] = r['final_tier']
    rec['final_pred_label'] = r['final_pred_label']
    rec['final_zone'] = r['final_zone']
    rec['true_label_name'] = r['true_label_name']
    rec['correct'] = r['correct']

    # Match rule by final tier
    tier = int(r['final_tier'])
    pred = r['final_pred_label'] if r['final_pred_label'] not in [np.nan, 'FP_NO_S2'] else '*'
    zone = r['final_zone'] if r['final_zone'] not in [np.nan, 'UNKNOWN'] else 'MEDIUM'

    rule = get_reflex_recommendation(pred, zone, tier, reflex_rules)
    if rule:
        rec['reflex_test'] = rule['test']
        rec['urgency'] = rule['urgency']
        rec['rationale'] = rule['rationale']
    else:
        rec['reflex_test'] = 'Manual review'
        rec['urgency'] = 'urgent'
        rec['rationale'] = 'No rule matched'

    reflex_rows.append(rec)

reflex_df = pd.DataFrame(reflex_rows)

# ─── Summary ───
print(f"═══ REFLEX TEST DISTRIBUTION ═══")
print(f"\nTest recommendation distribution:")
test_dist = reflex_df['reflex_test'].value_counts()
for test, count in test_dist.items():
    pct = count / len(reflex_df) * 100
    print(f"  {count:>4} ({pct:>5.1f}%) — {test}")

print(f"\nUrgency distribution:")
print(reflex_df['urgency'].value_counts())

print(f"\nTier × Urgency:")
print(pd.crosstab(reflex_df['final_tier'], reflex_df['urgency']))

print(f"\nCorrect/Incorrect × Urgency:")
print(pd.crosstab(reflex_df['correct'].map({True: 'Correct', False: 'Incorrect'}),
                  reflex_df['urgency']))

## CELL 14 — Add conformal sets to cascade_df + save

In [ ]:
# CELL 14 — Add conformal sets to cascade_df + save

# ─── Conformal matching ───
# S2 conformal: on s2_cbc index, match to cascade_df via s1_to_s2
if HAS_CONFORMAL:
    cs_s2_cbc = conformal['S2_CBC_Only']
    cs_s2_bio = conformal['S2_CBC_BIO']

    # Tier 1 conformal (S2 CBC_Only)
    t1_sets = []
    t1_sizes = []
    for _, r in cascade_df.iterrows():
        idx = r['patient_idx']
        s2_idx = s1_to_s2.get(idx)
        if r['tier1_zone'] not in ['Excluded'] and s2_idx is not None and s2_idx in cs_s2_cbc.index:
            in_cols = ['in_DEA', 'in_HA', 'in_HGB_HTZ', 'in_Normal']
            in_vals = cs_s2_cbc.loc[s2_idx, in_cols]
            members = [S2_LABELS[i] for i, c in enumerate(in_cols) if in_vals[c]]
            t1_sets.append('{' + ','.join(members) + '}' if members else '{}')
            t1_sizes.append(int(cs_s2_cbc.loc[s2_idx, 'set_size']))
        else:
            t1_sets.append(np.nan)
            t1_sizes.append(np.nan)

    cascade_df['tier1_conformal_set'] = t1_sets
    cascade_df['tier1_conformal_size'] = t1_sizes

    # Tier 2 conformal (S2 CBC_BIO)
    t2_sets = []
    t2_sizes = []
    for _, r in cascade_df.iterrows():
        idx = r['patient_idx']
        s2_idx = s1_to_s2.get(idx)
        if r['escalated'] and r.get('tier2_pred_label') not in ['DAS', 'FP_NO_S2', np.nan]:
            if s2_idx is not None and s2_idx in cs_s2_bio.index:
                in_cols = ['in_DEA', 'in_HA', 'in_HGB_HTZ', 'in_Normal']
                in_vals = cs_s2_bio.loc[s2_idx, in_cols]
                members = [S2_LABELS[i] for i, c in enumerate(in_cols) if in_vals[c]]
                t2_sets.append('{' + ','.join(members) + '}' if members else '{}')
                t2_sizes.append(int(cs_s2_bio.loc[s2_idx, 'set_size']))
            else:
                t2_sets.append(np.nan)
                t2_sizes.append(np.nan)
        else:
            t2_sets.append(np.nan)
            t2_sizes.append(np.nan)

    cascade_df['tier2_conformal_set'] = t2_sets
    cascade_df['tier2_conformal_size'] = t2_sizes

    print("═══ CONFORMAL INTEGRATION ═══")
    print(f"\nTier 1 conformal set size distribution:")
    print(cascade_df['tier1_conformal_size'].value_counts().sort_index())
    print(f"\nTier 2 conformal set size distribution:")
    print(cascade_df['tier2_conformal_size'].value_counts().sort_index())

    # Conformal size × accuracy
    for tier_label, size_col, correct_subset in [
        ('Tier 1 (HIGH only)', 'tier1_conformal_size', cascade_df[cascade_df['final_tier'] == 1]),
        ('Tier 2', 'tier2_conformal_size', cascade_df[cascade_df['final_tier'] == 2])
    ]:
        valid = correct_subset.dropna(subset=[size_col])
        if len(valid) > 0:
            print(f"\n{tier_label} — Set size × Accuracy:")
            for sz in sorted(valid[size_col].unique()):
                subset = valid[valid[size_col] == sz]
                acc = subset['correct'].mean()
                print(f"  Set size {int(sz)}: n={len(subset)}, acc={acc:.3f}")

else:
    cascade_df['tier1_conformal_set'] = np.nan
    cascade_df['tier1_conformal_size'] = np.nan
    cascade_df['tier2_conformal_set'] = np.nan
    cascade_df['tier2_conformal_size'] = np.nan
    print("⚠ No conformal data — filled with NaN")

# ─── Save cascade DataFrame ───
out_dir = PATHS.module_dir('m7_cascade', 'analysis')
cascade_df.to_parquet(out_dir / 'cascade_simulation.parquet', index=False)
print(f"\n✓ cascade_simulation.parquet saved: {out_dir}")
print(f"  Columns: {list(cascade_df.columns)}")
print(f"  Shape: {cascade_df.shape}")

## CELL 15 — Save efficiency curve data, reflex rules, metric tables

In [ ]:
# CELL 15 — Save efficiency curve data, reflex rules, metric tables
out_rules = PATHS.module_dir('m7_cascade', 'rules')
out_rules.mkdir(parents=True, exist_ok=True)
out_analysis = PATHS.module_dir('m7_cascade', 'analysis')
out_rules = PATHS.module_dir('m7_cascade', 'rules')

# 1. Efficiency curve data
sweep_df.to_parquet(out_analysis / 'efficiency_curve_data.parquet', index=False)
print(f"✓ efficiency_curve_data.parquet ({len(sweep_df)} rows)")

# 2. Reflex rules JSON
with open(out_rules / 'reflex_rules.json', 'w', encoding='utf-8') as f:
    json.dump(reflex_rules, f, ensure_ascii=False, indent=2)
print(f"✓ reflex_rules.json")

# 3. Reflex recommendations
reflex_df.to_parquet(out_analysis / 'reflex_recommendations.parquet', index=False)
print(f"✓ reflex_recommendations.parquet ({len(reflex_df)} rows)")

# 4. Cascade metric summary table
metrics = {
    'Metric': [
        'Total patients', 'Tier 1 HIGH', 'Tier 1 MEDIUM', 'Tier 1 LOW',
        'Tier 1 Excluded', 'Escalation rate',
        'Cascade E2E accuracy', 'Pure CBC_Only accuracy', 'Full BIO accuracy',
        'Cascade vs Full BIO gain', 'AAC lost in S1', 'AAC rescued in T2',
        'Reflex: routine', 'Reflex: urgent', 'Reflex: none'
    ],
    'Value': [
        len(cascade_df),
        (cascade_df['tier1_zone'] == 'HIGH').sum(),
        (cascade_df['tier1_zone'] == 'MEDIUM').sum(),
        (cascade_df['tier1_zone'] == 'LOW').sum(),
        (cascade_df['tier1_zone'] == 'Excluded').sum(),
        f"{cascade_df['escalated'].mean():.1%}",
        f"{cascade_df['correct'].mean():.3f}",
        '0.559', '0.699',
        f"{cascade_df['correct'].mean() - 0.699:+.3f}",
        (cascade_df['true_label'] != 99).sum() - (cascade_df[cascade_df['true_label'] != 99]['tier1_zone'] != 'Excluded').sum() + (cascade_df[(cascade_df['true_label'] != 99) & (cascade_df['tier1_zone'] == 'Excluded')].shape[0]),
        len(ias_escalated_correct),
        (reflex_df['urgency'] == 'routine').sum(),
        (reflex_df['urgency'] == 'urgent').sum(),
        (reflex_df['urgency'] == 'none').sum(),
    ]
}

metrics_df = pd.DataFrame(metrics)
metrics_df.to_excel(out_analysis / 'cascade_metrics.xlsx', index=False)
print(f"✓ cascade_metrics.xlsx")

# 5. Cost-effectiveness table
cost_eff = pd.DataFrame(results)  # from Cell 8v2
cost_eff.to_excel(out_analysis / 'cost_effectiveness.xlsx', index=False)
print(f"✓ cost_effectiveness.xlsx")

print(f"\n{'═'*50}")
print(f"All M7 outputs saved: {out_analysis.parent}")

## CELL 16 — Additional figures

In [ ]:
# CELL 16 — Additional figures

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

plot_dir = PATHS.module_dir('m7_cascade', 'plots')

# ─── Figure 2: Per-class loss rate ───
fig, ax = plt.subplots(figsize=(6, 4))

classes = ['DEA', 'HA', 'HGB_HTZ', 'Normal']           # internal keys (color lookup)
classes_disp = ['IDA', 'HA', 'HGB HTZ', 'Normal']       # display labels (x-axis)
ias_cas = cascade_df[cascade_df['true_label'] != 99]
loss_rates = []
for cls_id, cls_name in S2_LABELS.items():
    cls_df = ias_cas[ias_cas['true_label'] == cls_id]
    n_correct = (cls_df['correct'] == True).sum()
    loss_rates.append(1 - n_correct / len(cls_df) if len(cls_df) > 0 else 0)

colors = [CLASS_COLORS.get(c, PALETTE['base1']) for c in classes]
bars = ax.bar(classes_disp, [r * 100 for r in loss_rates], color=colors, edgecolor='white', linewidth=1.2)

for bar, rate in zip(bars, loss_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{rate:.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Loss Rate (%)', fontsize=11)
ax.set_xlabel('True Class', fontsize=11)
ax.set_ylim(0, 40)
ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
despine_all(ax)
fig.tight_layout()
save_fig(fig, plot_dir, 'class_loss_rate')
plt.show()

# ─── Figure 3: Conformal set size × accuracy (Tier 1 vs Tier 2) ───
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, (tier_label, size_col, tier_num) in zip(axes, [
    ('Tier 1 (CBC Only)', 'tier1_conformal_size', 1),
    ('Tier 2 (CBC+BIO)', 'tier2_conformal_size', 2)
]):
    subset = cascade_df[(cascade_df['final_tier'] == tier_num)].dropna(subset=[size_col])
    if len(subset) == 0:
        # Tier 1 HIGH all → has conformal
        # Tier 2 → escalated ones
        subset = cascade_df.dropna(subset=[size_col])
        if tier_num == 1:
            subset = subset[subset['escalated'] == False]

    sizes = sorted(subset[size_col].unique())
    accs = []
    counts = []
    for sz in sizes:
        s = subset[subset[size_col] == sz]
        accs.append(s['correct'].mean() * 100)
        counts.append(len(s))

    bars = ax.bar([str(int(s)) for s in sizes], accs,
                  color=PALETTE['accent1'], edgecolor='white', linewidth=1.2, alpha=0.8)

    for bar, acc, cnt in zip(bars, accs, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{acc:.0f}%\nn={cnt}', ha='center', va='bottom', fontsize=9)

    ax.set_title(tier_label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Conformal Set Size', fontsize=10)
    ax.set_ylim(0, 115)
    ax.yaxis.set_major_formatter(PercentFormatter(100, decimals=0))
    despine_all(ax)

axes[0].set_ylabel('Accuracy (%)', fontsize=11)
fig.tight_layout()
save_fig(fig, plot_dir, 'conformal_setsize_accuracy')
plt.show()

# ─── Figure 4: Reflex test urgency distribution ───
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: urgency by tier
urgency_colors = {'routine': PALETTE['accent1'], 'urgent': PALETTE['highlight'], 'none': PALETTE['base2']}

for ax, tier in zip(axes, [1, 2]):
    tier_data = reflex_df[reflex_df['final_tier'] == tier]
    urg_counts = tier_data['urgency'].value_counts()

    urg_labels = ['routine', 'urgent', 'none']
    urg_vals = [urg_counts.get(u, 0) for u in urg_labels]
    urg_cols = [urgency_colors[u] for u in urg_labels]

    bars = ax.bar(urg_labels, urg_vals, color=urg_cols, edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, urg_vals):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title(f'Tier {tier}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Patient Count' if tier == 1 else '', fontsize=10)
    despine_all(ax)

fig.suptitle('Reflex Test Urgency Distribution', fontsize=12, fontweight='bold', y=1.02)
fig.tight_layout()
save_fig(fig, plot_dir, 'reflex_urgency')
plt.show()

print(f"\n✓ All figures saved: {plot_dir}")

## CELL 17 — Create src/m7_cascade.py

In [ ]:
# CELL 17 — Create src/m7_cascade.py

m7_code = '''"""
M7 Cascade & Reflex Engine
===========================
Tier 1 (CBC_Only) → Tier 2 (CBC_BIO) cascade simulation,
efficiency curve, and reflex test rule engine.
"""

import json
import numpy as np
import pandas as pd
from pathlib import Path


S2_LABELS = {0: 'DEA', 1: 'HA', 2: 'HGB_HTZ', 3: 'Normal'}


def load_cascade_simulation(paths):
    """Load the saved cascade simulation."""
    fpath = paths.module_dir('m7_cascade', 'analysis') / 'cascade_simulation.parquet'
    return pd.read_parquet(fpath)


def load_efficiency_curve(paths):
    """Load the efficiency curve data."""
    fpath = paths.module_dir('m7_cascade', 'analysis') / 'efficiency_curve_data.parquet'
    return pd.read_parquet(fpath)


def load_reflex_rules(paths):
    """Load the reflex test rules."""
    fpath = paths.module_dir('m7_cascade', 'rules') / 'reflex_rules.json'
    with open(fpath, 'r', encoding='utf-8') as f:
        return json.load(f)


def load_reflex_recommendations(paths):
    """Load per-patient reflex recommendations."""
    fpath = paths.module_dir('m7_cascade', 'analysis') / 'reflex_recommendations.parquet'
    return pd.read_parquet(fpath)


def get_reflex_recommendation(pred_label, zone, tier, rules):
    """Find the most appropriate reflex test rule for a patient.

    Parameters
    ----------
    pred_label : str — predicted class (DEA, HA, HGB_HTZ, Normal, DAS)
    zone : str — confidence zone (HIGH, MEDIUM, LOW, Excluded)
    tier : int — cascade tier (1 or 2)
    rules : dict — reflex rule set (output of load_reflex_rules)

    Returns
    -------
    dict or None — matching rule
    """
    for rule in rules['rules']:
        if rule['tier'] != tier:
            continue
        zone_match = (rule['zone'] == zone)
        pred_match = (rule['prediction'] == pred_label or rule['prediction'] == '*')
        if zone_match and pred_match:
            return rule
    return None


def compute_e2e_pred(idx, s1_df, s2_df, ops, s1_to_s2):
    """Compute the E2E prediction for a single patient.

    Returns
    -------
    tuple: (pred_code, pred_label, confidence)
    """
    prob_ias = s1_df.loc[idx, 'prob_IAS_cal']
    s1_pred = int(prob_ias >= ops['s1_threshold'])

    if s1_pred == 0:
        return 99, 'DAS', 1 - prob_ias

    s2_idx = s1_to_s2.get(idx)
    if s2_idx is not None and s2_idx in s2_df.index:
        prob_cols = ['prob_DEA_cal', 'prob_HA_cal', 'prob_HGB_HTZ_cal', 'prob_Normal_cal']
        probs = s2_df.loc[s2_idx, prob_cols].values
        pred = int(np.argmax(probs))
        conf = float(np.max(probs))
        return pred, S2_LABELS[pred], conf
    else:
        return -1, 'FP_NO_S2', prob_ias


def build_s1_to_s2_mapping(s1_df, s2_df, feature_cols):
    """Feature-based index matching between S1 IAS patients and S2.

    Parameters
    ----------
    s1_df : DataFrame — full S1 data (not filtered to IAS)
    s2_df : DataFrame — S2 data (IAS patients only)
    feature_cols : list — common feature columns used for matching

    Returns
    -------
    dict: {s1_index: s2_index}
    """
    s1_ias = s1_df[s1_df['target'] == 1]
    s1_key = s1_ias[feature_cols].round(8).astype(str).agg('|'.join, axis=1)
    s2_key = s2_df[feature_cols].round(8).astype(str).agg('|'.join, axis=1)
    s2_key_to_idx = dict(zip(s2_key.values, s2_key.index))

    mapping = {}
    for s1_idx, key_val in s1_key.items():
        s2_idx = s2_key_to_idx.get(key_val)
        if s2_idx is not None:
            mapping[s1_idx] = s2_idx
    return mapping
'''

src_path = PATHS.src / 'm7_cascade.py'
src_path.write_text(m7_code, encoding='utf-8')
print(f"✓ {src_path} created ({len(m7_code)} characters)")